In [4]:
import matplotlib.pyplot as plt

from spectrum.spectrum2D import convNu2Ene, Spectrum2D
%matplotlib inline
import numpy as np
np.set_printoptions(precision=4)

import sys
sys.path.append("/home/vlew/CQCParse")

from CQCParse.parsing import GaussianDataParser
from CQCParse.relay import DataVault

from wilson import rendering
from wilson import analysis

from wilson.utils import get_package_root

wilson_root = get_package_root()
# data_vault = DataVault(wilson_root+'/tests/test_database/mini_files_database.csv')
data_vault = DataVault('/mnt/c/Users/vle014/OneDrive - UiT Office 365/Documents/files_fram/files_database.csv')

dataframe_gaussian = data_vault.getting_files_DB("gaussian")
method_basis = dataframe_gaussian[(dataframe_gaussian['code'] == 'ACAC') & (dataframe_gaussian['method'] != 'PBE0')][["code", "method", "basis_set"]]
tuples_method_basis = [(row['code'], row['method'], row['basis_set']) for index, row in method_basis.iterrows()]
# tuples_method_basis

log10=True
w1mw2=False
broad_factor_rc=10.

regions = {1: ((1282., 1457., 10.), (2589., 3051., 10.)),
           6: ((500., 3150., 10.), (500., 6050., 10.))}

# template
settings_here = {'electrical': [0, 1], 'mechanical': [0, 1, 2, 3, 4, 5],
                 'Gamma_rc': broad_factor_rc, 'region': 1,
                 'font_dict': {'size': 18}, 'figsize': (12, 15), 'norm_max': 1e12, 'norm_min': 1e7,
                 'dynamic_range_n': 30}

# terms_selection = [0, 1], [0, 1, 2, 3, 4, 5]
terms_selection = settings_here['electrical'], settings_here['mechanical'] # make a dictionary
region = 1

# omega1 = np.arange(*regions[region][0])
# omega2 = np.arange(*regions[region][1])
omega1 = [1164., 1175., 1186., 1314., 1324., 1334., 1362., 1372., 1382., 1776., 1786., 1796.]
omega2 = [1736., 1746., 1757., 1975., 1986., 1997., 2295., 2301., 2311., 2323., 2931., 2942., 2952., 3073., 3083., 3093.]

vibEL = False
datain = data_vault.make_DatainputDict('gaussian', ('FORM', 'B3LYP', 'cc_pVQZ'), '')

dictInputs = {'parserObject': GaussianDataParser(datain), 'el_terms_select': [], 'mech_terms_select': []}
spectrumObj = Spectrum2D(omega1, omega2)
spectrumObj.load_data(dictInputs['parserObject'])
spectrumObj.setSpectrumSettings(Gamma_rc=10., diag_margin_rc=10.)
# spectrumObj.conversion2InternalUnits() # need now at least for diag margin_rs in addTerms()
spectrumObj.addTerms(dictInputs['el_terms_select'], dictInputs['mech_terms_select'])
spectrumObj.precalculateParts()

# 'mu_Q', 'mu_QQ', 'alpha_Q', 'alpha_QQ', 'F_abc'
# spectrumObj.deriv_data['mu_Q'].shape
# spectrumObj.deriv_data['mu_QQ'].shape
# spectrumObj.deriv_data['alpha_Q'].shape
# spectrumObj.deriv_data['alpha_QQ'].shape


sec_hypol_data = 0
if settings_here['electrical']:
    electrical, Qab_contrib_dict = spectrumObj.intensity_electrical()
    sec_hypol_data += electrical

if settings_here['mechanical']:
    mechall, Qabc_contrib_dict = spectrumObj.intensity_mechanical()
    # template_array = np.full(mechall.shape, -0.+0.j, dtype=complex)
    sec_hypol_data += mechall
    
intensity = abs(sec_hypol_data) ** 2